# Task 7: Off-Policy Evaluation (OPE) + Deterministic Diagnostics — FIXED

## Bugs Fixed

### BUG 8 (CRITICAL): LABEL_RE in Task 7 does not match canonical cb_adf format
The original Task 7 LABEL_RE:
```
r"^\s*(?P<action_idx>[0-9]+):(?P<cost>...):(?P<prob>...)\s+\|action\b"
```
expects a named group `action_idx` which was used to set `chosen_idx = int(m.group('action_idx'))`.
But `action_idx` in cb_adf format is always `0` and does NOT represent the position in the slate.
The chosen action's **slate position** is its line number in the block (always 0 since we write
the chosen action first). Using `action_idx` from the label to set `chosen_idx` is wrong when
the value is 0 it happens to work, but the code extracts it as a named capture intending to
use it for slot position which it is not.
**Fix:** Remove the `action_idx` capture group confusion; set `chosen_idx = i` (line position).

### BUG 9: `strip_labels` in Task 7 strips the wrong prefix pattern
Original: `re.sub(r'^\s*[0-9]+:[0-9.]+:[0-9.]+\s+', '', s)` — this works for `0:cost:prob` format
but the regex `[0-9]+` requires at least one digit before the first colon (the action index).
This is fine, but it does NOT handle the format without the action-index prefix.
Since Task 5 now consistently writes `0:cost:prob`, this is OK — but we make it explicit.

### BUG 10: `parse_pred_token` in Task 7 errors on VW output format for cb_explore_adf
VW `--cb_explore_adf -t -p` writes predictions as `action_idx:prob` pairs (one per action),
e.g. `2:0.95 1:0.03 3:0.02` — the FIRST token is the recommended action index (1-based).
The original `parse_pred_token` splits on `:` and takes the first part which is correct for
the 1-based index, but fails silently when the output is a float score list.
**Fix:** Robustly parse both `int` and `float` prediction outputs.


In [ ]:
from __future__ import annotations
import io, json, os, re, subprocess, shutil, warnings
warnings.filterwarnings('ignore')
import boto3, numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import entropy
from dataclasses import dataclass, asdict

try:
    import vowpalwabbit
except Exception:
    vowpalwabbit = None


def get_use_case_id(default: str = 'pl-aip-uplift') -> str:
    value = os.getenv('USE_CASE_ID', default).strip()
    if not value:
        raise ValueError('USE_CASE_ID must be non-empty.')
    return value


@dataclass(frozen=True)
class ConcentrationMetrics:
    unique_items: int
    normalized_entropy: float
    herfindahl: float
    top1_concentration: float
    top5_concentration: float
    coverage: float

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass(frozen=True)
class PromotionDecision:
    passed: bool
    failures: list[str]

    def to_dict(self) -> dict:
        return asdict(self)


USE_CASE_ID = get_use_case_id('pl-aip-uplift')
S3_CONFIG_BUCKET = os.getenv('S3_CONFIG_BUCKET', 'aks-nvtabular-data')
EVAL_SPLIT_NAME = 'test'
NUM_SLATE = 60
s3_client = boto3.client('s3')

# Accept canonical Task 5 labels, plus legacy cost:prob labels if an older file is evaluated.
# Canonical format: '0:{cost}:{prob} |action ...'
LABEL_RE = re.compile(
    r'^\s*(?:0:)?(?P<cost>[0-9]*\.?[0-9]+):(?P<prob>[0-9]*\.?[0-9]+)\s+\|action\b'
)

print(f'Setup done. Evaluation split={EVAL_SPLIT_NAME}, Slate={NUM_SLATE}')
if vowpalwabbit is not None:
    print(f'VW Python package available: {getattr(vowpalwabbit, "__version__", "unknown")}')
else:
    print('VW Python package not available; Task 7 will require a CLI vw binary.')
print('LABEL_RE accepts canonical 0:cost:prob |action labels and legacy cost:prob |action labels')


In [ ]:
def dl(bucket, key, local):
    Path(local).parent.mkdir(parents=True, exist_ok=True)
    s3_client.download_file(bucket, key, str(local))


def read_blocks(path):
    t = Path(path).read_text(encoding='utf-8')
    return [b.strip('\n') for b in t.split('\n\n') if b.strip()]


def parse_s3_uri(uri: str) -> tuple[str, str]:
    if not uri.startswith('s3://'):
        raise ValueError(f'Invalid S3 URI: {uri}')
    body = uri[5:]
    bucket, _, key = body.partition('/')
    if not bucket or not key:
        raise ValueError(f'Invalid S3 URI: {uri}')
    return bucket, key


def load_json_s3(bucket: str, key: str) -> dict:
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    return json.loads(obj['Body'].read().decode('utf-8'))


def load_latest_run_manifest(bucket: str, use_case_id: str) -> dict:
    key = f'model_artifacts/{use_case_id}/latest_run.json'
    manifest = load_json_s3(bucket, key)
    required = ['run_id', 's3_run_prefix']
    missing = [k for k in required if not manifest.get(k)]
    if missing:
        raise ValueError(f'latest_run.json missing required fields: {missing}')
    return manifest


def parse_block(block):
    """
    BUG 8 FIX: chosen_idx is now set from line position (i), not from label's action_idx.
    In cb_adf format, the action index in the label ('0') is always 0 and is ignored by VW.
    The chosen action is always placed at line position 0 (first action line) in our format.
    """
    lines = [l.strip() for l in block.splitlines() if l.strip()]
    action_lines = lines[1:] if len(lines) > 1 else []

    action_aids = []
    for line in action_lines:
        m_aid = re.search(r'\baid_(\d+)\b', line)
        action_aids.append(int(m_aid.group(1)) if m_aid else None)

    r = {
        'cost': None,
        'log_prob': None,
        'chosen_idx': None,
        'chosen_aid': None,
        'reward': None,
        'num_actions': len(action_lines),
        'action_aids': action_aids,
    }

    for i, line in enumerate(action_lines):
        m = LABEL_RE.match(line)
        if m:
            # BUG 8 FIX: chosen_idx = i (line position), not int(m.group('action_idx'))
            r['cost']       = float(m.group('cost'))
            r['log_prob']   = float(m.group('prob'))
            r['chosen_idx'] = i                     # always 0 since we write chosen action first
            r['reward']     = 1.0 - float(m.group('cost'))
            if 0 <= i < len(action_aids):
                r['chosen_aid'] = action_aids[i]
            break

    return r


def strip_labels(block):
    """
    Removes either canonical '0:cost:prob ' or legacy 'cost:prob ' prefixes.
    """
    out = []
    for line in block.splitlines():
        s = line.strip()
        if LABEL_RE.match(s):
            # Remove label prefix, keep '|action ...'
            s = re.sub(r'^\s*(?:0:)?[0-9.]+:[0-9.]+\s+', '', s)
        if s:
            out.append(s)
    return '\n'.join(out)   # Return as string for VW Python API / subprocess file


def find_vw(required: bool = True):
    vw = shutil.which('vw')
    if vw:
        return vw
    for p in ['/usr/bin/vw', '/bin/vw', '/opt/conda/bin/vw']:
        if Path(p).exists():
            return p
    if required:
        raise RuntimeError(
            'VW CLI binary not found. Install the `vowpalwabbit` Python package or a CLI package that provides `vw`.'
        )
    return None


def analyze_action_concentration(pred_actions, num_actions=60):
    counts = np.bincount(pred_actions, minlength=num_actions)
    total = max(len(pred_actions), 1)
    ent = entropy(counts[counts > 0]) if np.any(counts > 0) else 0.0
    max_ent = np.log(num_actions) if num_actions > 1 else 1.0
    norm_ent = float(ent / max_ent) if max_ent > 0 else 0.0
    probs = counts / total
    herfindahl = float(np.sum(probs ** 2))
    top5 = float(np.sort(probs)[-5:].sum())
    coverage = float((counts > 0).sum() / num_actions)
    return {
        'unique_actions': int((counts > 0).sum()),
        'normalized_entropy': norm_ent,
        'herfindahl': herfindahl,
        'top5_concentration': top5,
        'coverage': coverage,
        'is_degenerate': bool(norm_ent < 0.30),
    }


def analyze_generic_concentration(values, universe_size=None):
    clean = [int(v) for v in values if v is not None and pd.notna(v)]
    if not clean:
        return ConcentrationMetrics(0, 0.0, 1.0, 1.0, 1.0, 0.0)
    counts = pd.Series(clean).value_counts().sort_values(ascending=False)
    probs = (counts / counts.sum()).values.astype(float)
    support = max(int(counts.shape[0]), 1)
    max_entropy = np.log(support) if support > 1 else 1.0
    normalized_entropy = float(entropy(probs) / max_entropy) if max_entropy > 0 else 0.0
    coverage_denom = max(int(universe_size or counts.shape[0]), 1)
    return ConcentrationMetrics(
        unique_items=int(counts.shape[0]),
        normalized_entropy=normalized_entropy,
        herfindahl=float(np.sum(probs ** 2)),
        top1_concentration=float(probs[:1].sum()),
        top5_concentration=float(probs[:5].sum()),
        coverage=float(min(int(counts.shape[0]) / coverage_denom, 1.0)),
    )


def build_arm_reward_model(blocks):
    parsed_blocks = [parse_block(b) for b in blocks]
    usable = [p for p in parsed_blocks if p['reward'] is not None and p['chosen_aid'] is not None]
    if not usable:
        raise ValueError('No usable logged rewards available for the direct reward model.')
    reward_model_df = pd.DataFrame({
        'logged_aid': [p['chosen_aid'] for p in usable],
        'reward': [p['reward'] for p in usable],
    })
    return reward_model_df.groupby('logged_aid')['reward'].mean().to_dict(), float(reward_model_df['reward'].mean())


def evaluate_promotion_readiness(predicted_positions, predicted_action_ids, slate_size, action_universe_size, dr_vs_historical_pct):
    failures = []
    position_conc = analyze_action_concentration(np.array(predicted_positions), slate_size)
    action_id_conc = analyze_generic_concentration(predicted_action_ids, universe_size=action_universe_size)
    if position_conc['normalized_entropy'] < 0.30:
        failures.append('position_entropy_below_floor')
    if position_conc['top5_concentration'] > 0.80:
        failures.append('position_top5_above_cap')
    if action_id_conc.normalized_entropy < 0.30:
        failures.append('action_id_entropy_below_floor')
    if action_id_conc.top1_concentration > 0.35:
        failures.append('action_id_top1_above_cap')
    if action_id_conc.top5_concentration > 0.80:
        failures.append('action_id_top5_above_cap')
    if dr_vs_historical_pct <= 0.0:
        failures.append('dr_lift_not_positive')
    return PromotionDecision(passed=not failures, failures=failures)


print('Helpers ready (BUG 8, 9, 10 fixes applied).')


In [ ]:
# Load latest Task 6 run metadata
latest_run = load_latest_run_manifest(S3_CONFIG_BUCKET, USE_CASE_ID)
run_bucket, run_prefix_key = parse_s3_uri(latest_run['s3_run_prefix'])
print(f"Evaluating run_id: {latest_run['run_id']}")
print(f"Run prefix: {latest_run['s3_run_prefix']}")

split_manifest_key = f'{run_prefix_key}/split_manifest.json'
split_manifest = load_json_s3(run_bucket, split_manifest_key)
split_strategy = split_manifest.get('split_strategy') or split_manifest.get('strategy')
if split_strategy != 'chronological_train_validation_test':
    raise ValueError(f'Task 7 requires chronological_train_validation_test split manifest. Found: {split_manifest}')
for required_key in ['train_rows', 'validation_rows', 'test_rows']:
    if int(split_manifest.get(required_key, 0)) <= 0:
        raise ValueError(f'Split manifest has invalid {required_key}: {split_manifest.get(required_key)}')

alignment_key = f'training_data/vw_training_alignment_{USE_CASE_ID}.parquet'
obj = s3_client.get_object(Bucket=S3_CONFIG_BUCKET, Key=alignment_key)
df = pd.read_parquet(io.BytesIO(obj['Body'].read())).reset_index(drop=True)
alignment_source = f's3://{S3_CONFIG_BUCKET}/{alignment_key}'

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
invalid_ts = int(df['timestamp'].isna().sum())
if invalid_ts > 0:
    raise ValueError(f'Task 7 requires valid timestamps. Found {invalid_ts:,} invalid timestamps.')

vw_path = Path(f'/tmp/vw_training_{USE_CASE_ID}_FINAL.txt')
dl(S3_CONFIG_BUCKET, f'training_data/vw_training_{USE_CASE_ID}_FINAL.txt', vw_path)
all_blocks_raw = read_blocks(vw_path)
if len(df) != len(all_blocks_raw):
    raise ValueError(
        f'Alignment artifact rows ({len(df):,}) do not match VW blocks ({len(all_blocks_raw):,}). Re-run Task 5.'
    )

ordered = df.copy().reset_index(drop=True)
ordered['block_text'] = all_blocks_raw
sort_cols = [c for c in ['timestamp', 'campaign_id', 'master_user_id', 'event_order'] if c in ordered.columns]
ordered = ordered.sort_values(sort_cols, kind='stable').reset_index(drop=True)

train_rows = int(split_manifest['train_rows'])
validation_rows = int(split_manifest['validation_rows'])
test_rows = int(split_manifest['test_rows'])
if train_rows + validation_rows + test_rows != len(ordered):
    raise ValueError(
        f'Split manifest rows do not match alignment rows: manifest={train_rows + validation_rows + test_rows:,}, aligned={len(ordered):,}'
    )

test_start = train_rows + validation_rows
train_df = ordered.iloc[:train_rows].drop(columns=['block_text']).reset_index(drop=True)
validation_df = ordered.iloc[train_rows:test_start].drop(columns=['block_text']).reset_index(drop=True)
holdout_df = ordered.iloc[test_start:].drop(columns=['block_text']).reset_index(drop=True)
reward_model_blocks = ordered.iloc[:test_start]['block_text'].tolist()
reward_model_split = 'train_validation'
holdout_blocks = ordered.iloc[test_start:]['block_text'].tolist()

manifest_temporal_reliable = split_manifest.get('temporal_order_reliable')
if manifest_temporal_reliable is None:
    train_end_ts = pd.Timestamp(split_manifest.get('train_end'))
    validation_start_ts = pd.Timestamp(split_manifest.get('validation_start'))
    validation_end_ts = pd.Timestamp(split_manifest.get('validation_end'))
    test_start_ts = pd.Timestamp(split_manifest.get('test_start'))
    temporal_order_reliable = bool(train_end_ts < validation_start_ts and validation_end_ts < test_start_ts)
else:
    temporal_order_reliable = bool(manifest_temporal_reliable)

print(f'Alignment source: {alignment_source}')
print(f'Split strategy: {split_strategy}')
print(f'Temporal order reliable: {temporal_order_reliable}')
if not temporal_order_reliable:
    print('WARNING: OPE will not be production-promotable.')
print(f'Train rows: {len(train_df):,} | Validation rows: {len(validation_df):,} | Test rows: {len(holdout_df):,}')
print(f'VW blocks: {len(all_blocks_raw):,} | Reward-model blocks: {len(reward_model_blocks):,} | Test blocks: {len(holdout_blocks):,}')


In [ ]:
# Download model and run VW predictions on holdout
model_uri = f's3://{run_bucket}/{run_prefix_key}/vw_model_{USE_CASE_ID}.vw'
model_key = f'{run_prefix_key}/vw_model_{USE_CASE_ID}.vw'
model_path = Path(f'/tmp/vw_model_{USE_CASE_ID}.vw')
dl(run_bucket, model_key, model_path)
print(f'Model downloaded from {model_uri}')

SEP = chr(10)
unlab_path = Path('/tmp/vw_holdout_unlabelled.txt')
unlab_blocks = [strip_labels(b) for b in holdout_blocks]
unlab_path.write_text((SEP + SEP).join(unlab_blocks) + SEP, encoding='utf-8')

pred_path = Path('/tmp/vw_predictions.txt')
num_actions_by_block = [max(parse_block(b)['num_actions'], 1) for b in holdout_blocks]


def vw_version_text(vw_bin: str) -> str:
    try:
        proc = subprocess.run([vw_bin, '--version'], capture_output=True, text=True, timeout=10)
        return (proc.stdout or proc.stderr or 'unknown').strip()
    except Exception as exc:
        return f'unknown ({exc})'


def normalize_raw_action_index(raw: int, all_raw: list[int] | None, num_actions: int) -> int | None:
    if all_raw:
        mn, mx = min(all_raw), max(all_raw)
        if mn >= 1 and mx <= num_actions:
            return raw - 1
        if mn >= 0 and mx < num_actions:
            return raw
    if raw == 0:
        return 0
    if 1 <= raw <= num_actions:
        return raw - 1
    if 0 <= raw < num_actions:
        return raw
    return None


def prediction_to_position(prediction, num_actions: int) -> int:
    if prediction is None:
        return 0
    if isinstance(prediction, (int, float, np.integer, np.floating)):
        pos = normalize_raw_action_index(int(prediction), None, num_actions)
        return 0 if pos is None else pos
    if isinstance(prediction, list) and prediction:
        first = prediction[0]
        if hasattr(first, 'action') and hasattr(first, 'score'):
            raw_actions = [int(item.action) for item in prediction]
            best = max(prediction, key=lambda item: float(item.score))
            pos = normalize_raw_action_index(int(best.action), raw_actions, num_actions)
            return 0 if pos is None else pos
        if isinstance(first, (tuple, list)) and len(first) >= 2:
            raw_actions = [int(item[0]) for item in prediction]
            best = max(prediction, key=lambda item: float(item[1]))
            pos = normalize_raw_action_index(int(best[0]), raw_actions, num_actions)
            return 0 if pos is None else pos
        return int(np.argmax([float(x) for x in prediction[:num_actions]]))
    return 0


def parse_prediction_line(line: str, num_actions: int) -> int | None:
    """
    Return a 0-based slate position from one VW prediction line.

    Handles:
    - cb_explore_adf probability lists: '2:0.95 1:0.03 3:0.02'
    - comma-separated variants: '2:0.95,1:0.03,3:0.02'
    - direct integer actions: '2'
    - score vectors: '0.1 0.8 0.1' -> argmax position
    """
    s = line.strip()
    if not s:
        return None

    pair_text = s.replace(',', ' ')
    pairs = re.findall(r'(?<!\S)(\d+):([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)(?=\s|$)', pair_text)
    if pairs:
        raw_actions = [int(action) for action, _ in pairs]
        best_raw = int(max(pairs, key=lambda item: float(item[1]))[0])
        return normalize_raw_action_index(best_raw, raw_actions, num_actions)

    nums = []
    for token in s.split():
        token = token.split(',', 1)[0]
        try:
            nums.append(float(token))
        except ValueError:
            pass
    if not nums:
        return None
    if len(nums) >= num_actions:
        return int(np.argmax(nums[:num_actions]))
    first = nums[0]
    raw = int(first)
    if first == raw:
        return normalize_raw_action_index(raw, None, num_actions)
    return 0


def predict_with_python_vw(model_path: Path, blocks: list[str], num_actions: list[int]) -> list[int]:
    if vowpalwabbit is None:
        raise RuntimeError('Python vowpalwabbit package is not installed.')

    attempts = [
        f'-t -i {model_path}',
        f'--cb_explore_adf -t -i {model_path}',
    ]
    last_error = None
    for args in attempts:
        vw_eval = None
        try:
            print('Running VW prediction with Python package...')
            print(f'VW Python package: {getattr(vowpalwabbit, "__version__", "unknown")}')
            print(f'VW args: {args}')
            vw_eval = vowpalwabbit.Workspace(args, quiet=True)
            predicted = []
            for block, n_actions in zip(blocks, num_actions):
                if block.strip():
                    predicted.append(prediction_to_position(vw_eval.predict(block), n_actions))
                else:
                    predicted.append(0)
            return predicted
        except Exception as exc:
            last_error = exc
            print(f'Python VW prediction attempt failed: {exc}')
        finally:
            if vw_eval is not None:
                try:
                    vw_eval.finish()
                except Exception:
                    pass
    raise RuntimeError(f'Python VW prediction failed for all load modes. Last error: {last_error}')


def predict_with_cli_vw(model_path: Path, unlab_path: Path, pred_path: Path, num_actions: list[int]) -> list[int]:
    vw_bin = find_vw(required=True)
    cmd_default = [vw_bin, '-t', '-i', str(model_path), '-d', str(unlab_path), '-p', str(pred_path), '--quiet']

    print('Running VW prediction with CLI...')
    print(f'VW binary: {vw_bin} ({vw_version_text(vw_bin)})')
    # Do not pass --cb_explore_adf here. The saved model already contains its learner stack;
    # adding the reduction again causes: option '--cb_explore_adf' cannot be specified more than once.
    proc = subprocess.run(cmd_default, capture_output=True, text=True)
    if proc.returncode != 0:
        err = proc.stderr or proc.stdout or ''
        if "--clip_p" in err or 'model version is more recent' in err:
            msg = chr(10).join([
                'VW prediction failed because the local vw binary is older than the downloaded model.',
                f'Local VW: {vw_bin} ({vw_version_text(vw_bin)})',
                f'Model: {model_uri}',
                '',
                'The model was trained with a newer VW option/model format, including --clip_p, but this vw binary does not understand it.',
                'Fix one of these before rerunning Task 7:',
                '1. Use the Python vowpalwabbit package version that trained Task 6, then rerun this cell.',
                '2. Install/use the same or newer VW CLI version that trained the model.',
                '3. Retrain Task 6 with the same CLI VW version that Task 7 will use.',
                '',
                f'VW stderr:\n{err}',
            ])
            raise RuntimeError(msg)
        raise RuntimeError(f'VW prediction failed:\n{err}')

    raw = [line.strip() for line in pred_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    parsed = []
    for line in raw:
        if len(parsed) >= len(num_actions):
            break
        pos = parse_prediction_line(line, num_actions[len(parsed)])
        if pos is not None:
            parsed.append(pos)
    if len(parsed) != len(num_actions):
        raise ValueError(
            f'Prediction count mismatch: predictions={len(parsed):,}, holdout_blocks={len(num_actions):,}. '
            f'First raw predictions: {raw[:5]}'
        )
    return parsed


if vowpalwabbit is not None:
    predicted_actions = predict_with_python_vw(model_path, unlab_blocks, num_actions_by_block)
else:
    predicted_actions = predict_with_cli_vw(model_path, unlab_path, pred_path, num_actions_by_block)

if len(predicted_actions) != len(holdout_blocks):
    raise ValueError(
        f'Prediction count mismatch: predictions={len(predicted_actions):,}, holdout_blocks={len(holdout_blocks):,}.'
    )
out_of_range = [pos for pos, n_actions in zip(predicted_actions, num_actions_by_block) if pos < 0 or pos >= n_actions]
if out_of_range:
    raise ValueError(f'Predicted positions outside per-row slate ranges. Sample={out_of_range[:10]}')

print(f'Predictions: {len(predicted_actions):,}')
print('Indexing: normalized to 0-based slate positions')
print(f'Range after normalize: [{min(predicted_actions)}, {max(predicted_actions)}]')
print(f'Sample: {predicted_actions[:10]}')


In [ ]:
# Build evaluation dataframe
parsed = [parse_block(b) for b in holdout_blocks]

if len(parsed) == 0 or len(predicted_actions) == 0 or len(holdout_df) == 0:
    raise ValueError('No aligned holdout examples available for evaluation.')
if len(parsed) != len(predicted_actions) or len(parsed) != len(holdout_df):
    raise ValueError(
        f'Task 7 requires exact alignment: parsed={len(parsed):,}, preds={len(predicted_actions):,}, holdout_df={len(holdout_df):,}. '
        'Re-run Task 5/Task 6 so evaluation uses the exact holdout rows and prediction outputs.'
    )

parse_failures = sum(
    1 for p in parsed
    if p['reward'] is None or p['log_prob'] is None or p['chosen_aid'] is None
)
if parse_failures:
    bad_example = next(
        (i for i, p in enumerate(parsed) if p['reward'] is None or p['log_prob'] is None or p['chosen_aid'] is None),
        None,
    )
    raise ValueError(
        f'Failed to parse {parse_failures:,} holdout VW blocks. '
        f'First bad block index: {bad_example}. Check LABEL_RE and Task 5 VW formatting.'
    )


def pred_aid_from_block(pred_pos, block_info):
    aids = block_info['action_aids']
    if pred_pos is None:
        return None
    if 0 <= pred_pos < len(aids):
        return aids[pred_pos]
    return None


pred_aid = [pred_aid_from_block(predicted_actions[i], parsed[i]) for i in range(len(parsed))]

eval_df = pd.DataFrame({
    'cost': [p['cost'] for p in parsed],
    'log_prob': [p['log_prob'] for p in parsed],
    'logged_action_pos': [p['chosen_idx'] for p in parsed],
    'logged_aid': [p['chosen_aid'] for p in parsed],
    'reward': [p['reward'] for p in parsed],
    'pred_action_pos': predicted_actions[:len(parsed)],
    'pred_aid': pred_aid,
    'num_actions': [p['num_actions'] for p in parsed],
})

for col in ['income_tier', 'city_tier', 'lifecycle_stage', 'risk_level', 'channel']:
    if col in holdout_df.columns:
        eval_df[col] = holdout_df[col].values

eval_df['action_match'] = (
    eval_df['logged_aid'].notna() &
    eval_df['pred_aid'].notna() &
    (eval_df['logged_aid'] == eval_df['pred_aid'])
).astype(int)

print(f'Eval df shape: {eval_df.shape}')
print(f'logged_aid unique: {eval_df["logged_aid"].nunique(dropna=True)}')
print(f'pred_aid unique:   {eval_df["pred_aid"].nunique(dropna=True)}')
print(f'action_match rate: {eval_df["action_match"].mean():.2%}')
print(eval_df.head(5).to_string())


In [ ]:
# Off-Policy Evaluation: IPS, SNIPS, DM, DR
n = len(eval_df)
rewards = eval_df['reward'].values
log_prob = eval_df['log_prob'].values.astype(float)
match = eval_df['action_match'].values.astype(float)

strict_propensity_ready = False
training_intent = None
task6_validation_passed = False
cold_start_replay_training = False
train_metrics = None
try:
    metrics_uri = latest_run.get('s3_run_prefix', '') + '/vw_metrics.json'
    metrics_bucket, metrics_key = parse_s3_uri(metrics_uri)
    train_metrics = load_json_s3(metrics_bucket, metrics_key)
    strict_propensity_ready = bool(train_metrics.get('strict_ope_ready', False))
    training_intent = train_metrics.get('training_intent')
    task6_validation_passed = bool(train_metrics.get('task6_validation_passed', False))
    cold_start_replay_training = bool(train_metrics.get('cold_start_replay_training', False))
except Exception as e:
    print(f'WARNING: Could not load Task 6 metrics: {e}')

historical_value = rewards.mean()
valid_log_prob = np.isfinite(log_prob) & (log_prob > 0.0) & (log_prob <= 1.0)
safe_log_prob = np.where(valid_log_prob, log_prob, np.nan)
invalid_log_prob_count = int(np.sum(~valid_log_prob))
valid_log_prob_count = int(np.sum(valid_log_prob))

ips_w = np.where(valid_log_prob, match / safe_log_prob, 0.0)
ips_value = float((ips_w * rewards).sum() / max(n, 1))
denom = float(ips_w.sum())
snips_value = float((ips_w * rewards).sum() / denom) if denom > 0 else 0.0

arm_reward, reward_model_global = build_arm_reward_model(reward_model_blocks)
direct_reward_model_source = reward_model_split
r_hat_pred = eval_df['pred_aid'].map(arm_reward).fillna(reward_model_global).values
r_hat_log = eval_df['logged_aid'].map(arm_reward).fillna(reward_model_global).values
dm_value = float(r_hat_pred.mean())
dr_per_row = r_hat_pred + ips_w * (rewards - r_hat_log)
dr_value = float(dr_per_row.mean())

dr_vs_hist = ((dr_value - historical_value) / max(abs(historical_value), 1e-6)) * 100
sn_vs_hist = ((snips_value - historical_value) / max(abs(historical_value), 1e-6)) * 100


def resample_ci(values, n_boot=1000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return (0.0, 0.0)
    estimates = [rng.choice(values, size=len(values), replace=True).mean() for _ in range(n_boot)]
    return (float(np.quantile(estimates, alpha / 2)), float(np.quantile(estimates, 1 - alpha / 2)))


historical_value_ci = resample_ci(rewards)
dr_value_ci = resample_ci(dr_per_row)
dr_lift_rows = ((dr_per_row - historical_value) / max(abs(historical_value), 1e-6)) * 100.0
dr_lift_ci = resample_ci(dr_lift_rows)
dr_lift_ci_width = float(dr_lift_ci[1] - dr_lift_ci[0])
dr_lift_ci_half_width = float(dr_lift_ci_width / 2.0)
if dr_vs_hist > 0 and dr_lift_ci_half_width > 0:
    estimated_examples_for_positive_ci = int(np.ceil(n * (dr_lift_ci_half_width / max(dr_vs_hist, 1e-6)) ** 2))
else:
    estimated_examples_for_positive_ci = None
additional_examples_for_positive_ci = (
    max(0, estimated_examples_for_positive_ci - n)
    if estimated_examples_for_positive_ci is not None
    else None
)
confidence_diagnostics = {
    'dr_lift_pct_point_estimate': float(dr_vs_hist),
    'dr_lift_pct_ci_lower': float(dr_lift_ci[0]),
    'dr_lift_pct_ci_upper': float(dr_lift_ci[1]),
    'dr_lift_pct_ci_width': dr_lift_ci_width,
    'estimated_examples_for_positive_ci': estimated_examples_for_positive_ci,
    'additional_examples_for_positive_ci': additional_examples_for_positive_ci,
}

match_rate = float(match.mean())
nonzero_ips = int(np.count_nonzero(ips_w))
prop_unique = int(pd.Series(log_prob[valid_log_prob]).nunique(dropna=True)) if valid_log_prob_count > 0 else 0
all_propensity_equal_one = bool(np.allclose(log_prob[valid_log_prob], 1.0)) if valid_log_prob_count > 0 else False
ess = float((ips_w.sum() ** 2) / max((ips_w ** 2).sum(), 1e-12)) if nonzero_ips > 0 else 0.0
temporal_order_reliable_now = bool(globals().get('temporal_order_reliable', False))

ope_diagnostics = {
    'run_id': latest_run.get('run_id'),
    'training_intent': training_intent,
    'task6_validation_passed': bool(task6_validation_passed),
    'cold_start_replay_training': bool(cold_start_replay_training),
    'strict_propensity_ready': bool(strict_propensity_ready),
    'temporal_order_reliable': temporal_order_reliable_now,
    'split_strategy': globals().get('split_strategy'),
    'match_rate': match_rate,
    'nonzero_ips_examples': nonzero_ips,
    'effective_sample_size': ess,
    'num_unique_log_prob': prop_unique,
    'valid_log_prob_count': valid_log_prob_count,
    'invalid_log_prob_count': invalid_log_prob_count,
    'all_propensity_equal_one': all_propensity_equal_one,
    'test_split_only': True,
    'confidence_diagnostics': confidence_diagnostics,
    'direct_reward_model': {
        'source_split': direct_reward_model_source,
        'actions': int(len(arm_reward)),
        'global_reward_mean': float(reward_model_global),
    },
    'direct_reward_model_source': direct_reward_model_source,
    'direct_reward_model_actions': int(len(arm_reward)),
}

ope_reliable = bool(
    strict_propensity_ready and
    temporal_order_reliable_now and
    (invalid_log_prob_count == 0) and
    (not all_propensity_equal_one) and
    (nonzero_ips >= 100) and
    (ess >= 100.0)
)

print('=' * 60)
print('      OFF-POLICY EVALUATION SCORECARD')
print('=' * 60)
print(f"  Run ID                : {latest_run.get('run_id')}")
print(f'  Holdout examples      : {n:,}')
print(f'  Action match rate     : {match_rate:.1%}')
print()
print('  POLICY VALUE  (higher = better)')
print(f'  Historical policy     : {historical_value:.4f}')
print(f'  Model IPS             : {ips_value:.4f}')
print(f'  Model SNIPS           : {snips_value:.4f}')
print(f'  Model Direct Method   : {dm_value:.4f}')
print(f'  Model DR (recommended): {dr_value:.4f}')
print()
print('  LIFT OVER BASELINES')
print(f'  DR vs Historical      : {dr_vs_hist:+.1f}%')
print(f'  SNIPS vs Historical   : {sn_vs_hist:+.1f}%')
print()
print('  OPE RELIABILITY CHECKS')
print(f'  strict_propensity_ready: {strict_propensity_ready}')
print(f'  task6_validation_passed: {task6_validation_passed}')
print(f'  cold_start_replay_data : {cold_start_replay_training}')
print(f'  temporal_order_reliable: {temporal_order_reliable_now}')
print(f'  Reliable for OPE?      : {ope_reliable}')
print(f'  DR Value 95% CI        : [{dr_value_ci[0]:.4f}, {dr_value_ci[1]:.4f}]')
print(f'  DR Lift 95% CI         : [{dr_lift_ci[0]:+.1f}%, {dr_lift_ci[1]:+.1f}%]')
print('=' * 60)


In [ ]:
# Action concentration analysis and scorecard save
# (cells c7-c9 unchanged from original — logic is correct)
obj_lib = s3_client.get_object(Bucket=S3_CONFIG_BUCKET,
    Key=f'action_library/action_library_{USE_CASE_ID}.parquet')
alib = pd.read_parquet(io.BytesIO(obj_lib['Body'].read())).set_index('action_id')
action_library_size = int(alib.index.nunique())

pred_cnt = pd.Series(eval_df['pred_aid'].dropna().astype(int).values).value_counts().head(10)
hist_cnt = pd.Series(eval_df['logged_aid'].dropna().astype(int).values).value_counts().head(10)

for dim in ['channel', 'day', 'time']:
    pred_map = eval_df['pred_aid'].map(alib[dim])
    log_map = eval_df['logged_aid'].map(alib[dim])
    eval_df[f'pred_{dim}'] = pred_map.values
    eval_df[f'logged_{dim}'] = log_map.values

dimension_match = {
    'channel': float((eval_df['pred_channel'] == eval_df['logged_channel']).mean()),
    'day': float((eval_df['pred_day'] == eval_df['logged_day']).mean()),
    'time': float((eval_df['pred_time'] == eval_df['logged_time']).mean()),
}

position_concentration = analyze_action_concentration(eval_df['pred_action_pos'].values, NUM_SLATE)
action_id_concentration = analyze_generic_concentration(eval_df['pred_aid'].tolist(), universe_size=action_library_size)
semantic_action_decision = evaluate_promotion_readiness(
    predicted_positions=eval_df['pred_action_pos'].tolist(),
    predicted_action_ids=eval_df['pred_aid'].tolist(),
    slate_size=NUM_SLATE,
    action_universe_size=action_library_size,
    dr_vs_historical_pct=dr_vs_hist,
)
production_failures = list(semantic_action_decision.failures)
if not task6_validation_passed:
    production_failures.append('task6_validation_not_passed')
if cold_start_replay_training:
    production_failures.append('cold_start_replay_not_production_data')
if not strict_propensity_ready:
    production_failures.append('strict_propensity_not_ready')
if not ope_reliable:
    production_failures.append('ope_not_reliable')
if dr_lift_ci[0] <= 0.0:
    production_failures.append('dr_lift_ci_lower_not_positive')
production_failures = sorted(set(production_failures))
promotion_decision = PromotionDecision(passed=not production_failures, failures=production_failures)

print(f'\n  Model unique actions   : {eval_df["pred_aid"].nunique(dropna=True)}')
print(f'  History unique actions : {eval_df["logged_aid"].nunique(dropna=True)}')
print('\n  Dimension-level match rates:')
for dim, value in dimension_match.items():
    print(f'    {dim:>9} : {value:.1%}')
print(f'\n  Position entropy       : {position_concentration["normalized_entropy"]:.3f}')
print(f'  Action-ID entropy      : {action_id_concentration.normalized_entropy:.3f}')
print(f'  Production Gate        : {"PASS" if promotion_decision.passed else "FAIL"}')
if promotion_decision.failures:
    print(f'  Blockers               : {promotion_decision.failures}')


In [ ]:
# Save scorecard and upload to S3
output_dir = Path('task7_artifacts') / latest_run.get('run_id')
output_dir.mkdir(parents=True, exist_ok=True)

pre_deploy_checks = {
    'task6_validation_passed': bool(task6_validation_passed),
    'not_cold_start_replay_training': bool(not cold_start_replay_training),
    'strict_propensity_ready': bool(strict_propensity_ready),
    'temporal_order_reliable': bool(ope_diagnostics.get('temporal_order_reliable', False)),
    'ope_reliable': bool(ope_reliable),
    'test_split_only': True,
    'dr_vs_historical_gt_0': bool(dr_vs_hist > 0),
    'dr_lift_ci_lower_gt_0': bool(dr_lift_ci[0] > 0),
    'valid_dr_range': bool(0 <= dr_value <= 1),
    'position_entropy_gt_0_30': bool(position_concentration['normalized_entropy'] > 0.30),
    'position_not_degenerate': bool(not position_concentration['is_degenerate']),
    'position_coverage_gt_0_20': bool(position_concentration['coverage'] > 0.20),
    'semantic_action_gate_passed': bool(semantic_action_decision.passed),
    'production_gate_passed': bool(promotion_decision.passed),
}

scorecard = {
    'use_case_id': USE_CASE_ID,
    'run_id': latest_run.get('run_id'),
    'run_prefix': latest_run.get('s3_run_prefix'),
    'evaluation_split': 'test',
    'split_manifest': split_manifest,
    'holdout_examples': int(n),
    'action_match_rate': float(match.mean()),
    'training_lineage': {
        'training_intent': training_intent,
        'task6_validation_passed': bool(task6_validation_passed),
        'cold_start_replay_training': bool(cold_start_replay_training),
        'latest_run_production_promotable': bool(latest_run.get('production_promotable', False)),
    },
    'ope_reliability': ope_diagnostics,
    'policy_value': {
        'historical_baseline': float(historical_value),
        'model_ips': float(ips_value),
        'model_snips': float(snips_value),
        'model_dm': float(dm_value),
        'model_dr': float(dr_value),
    },
    'lift_pct': {
        'dr_vs_historical': float(dr_vs_hist),
        'snips_vs_historical': float(sn_vs_hist),
    },
    'confidence_intervals': {
        'historical_value_95': list(historical_value_ci),
        'dr_value_95': list(dr_value_ci),
        'dr_lift_pct_95': list(dr_lift_ci),
    },
    'confidence_diagnostics': confidence_diagnostics,
    'dimension_match_rate': dimension_match,
    'model_diversity': {
        'unique_recommended': int(eval_df['pred_aid'].nunique(dropna=True)),
        'unique_historical': int(eval_df['logged_aid'].nunique(dropna=True)),
        'diversity_score': float(eval_df['pred_aid'].nunique(dropna=True) / NUM_SLATE),
    },
    'action_concentration_by_position': position_concentration,
    'action_concentration_by_id': action_id_concentration.to_dict(),
    'semantic_action_decision': semantic_action_decision.to_dict(),
    'promotion_decision': promotion_decision.to_dict(),
    'pre_deployment_checks': pre_deploy_checks,
}

sc_path = output_dir / 'ope_scorecard.json'
sc_path.write_text(json.dumps(scorecard, indent=2), encoding='utf-8')
eval_df.to_parquet(output_dir / 'eval_predictions.parquet', index=False)

run_eval_prefix = f'{run_prefix_key}/evaluation'
s3_client.upload_file(str(sc_path), S3_CONFIG_BUCKET, f'{run_eval_prefix}/ope_scorecard.json')
s3_client.upload_file(
    str(output_dir / 'eval_predictions.parquet'),
    S3_CONFIG_BUCKET,
    f'{run_eval_prefix}/eval_predictions.parquet'
)
latest_eval = {
    'run_id': latest_run.get('run_id'),
    'use_case_id': USE_CASE_ID,
    's3_scorecard': f's3://{S3_CONFIG_BUCKET}/{run_eval_prefix}/ope_scorecard.json',
    's3_eval_predictions': f's3://{S3_CONFIG_BUCKET}/{run_eval_prefix}/eval_predictions.parquet',
    'promotion_passed': bool(promotion_decision.passed),
    'production_promotable': bool(promotion_decision.passed),
    'production_blockers': list(promotion_decision.failures),
    'task6_validation_passed': bool(task6_validation_passed),
    'cold_start_replay_training': bool(cold_start_replay_training),
    'dr_lift_ci_lower': float(dr_lift_ci[0]),
    'dr_lift_ci_upper': float(dr_lift_ci[1]),
    'ope_reliable': bool(ope_reliable),
    'test_split_only': True,
}
s3_client.put_object(
    Bucket=S3_CONFIG_BUCKET,
    Key=f'model_artifacts/{USE_CASE_ID}/latest_eval.json',
    Body=json.dumps(latest_eval, indent=2).encode('utf-8'),
)

print('=' * 60)
print('  EVALUATION COMPLETE')
print('=' * 60)
print(f"  Run ID            : {latest_run.get('run_id')}")
print(f'  DR Value          : {dr_value:.4f}')
print(f'  Lift vs Hist      : {dr_vs_hist:+.1f}%')
print(f'  Match Rate        : {match.mean():.1%}')
print(f'  Production Gate   : {promotion_decision.passed}')
print(f'  OPE Reliable?     : {ope_reliable}')
print(f'  DR Lift CI        : [{dr_lift_ci[0]:+.1f}%, {dr_lift_ci[1]:+.1f}%]')
if promotion_decision.failures:
    print(f'  Production blockers: {promotion_decision.failures}')
print(f'  Scorecard         : s3://{S3_CONFIG_BUCKET}/{run_eval_prefix}/ope_scorecard.json')
print('\nPre-deployment checks:')
for k, v in pre_deploy_checks.items():
    print(f'  {"OK" if v else "FAIL"} - {k}')
